# Streaming

<img src="./assets/LC_streaming.png" width="400">

Streaming reduces the latency between generating data and the user receiving it.
There are two types frequently used with Agents:

Let's start by setting up a basic agent to demonstrate different streaming approaches.


In [11]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";

const agent = createAgent({
    model: process.env.AI_MODEL || "anthropic:claude-sonnet-4-6",
    systemPrompt: "You are a full-stack comedian",
})

Now that we have our agent configured, let's first see how it works without streaming.


## No Streaming (invoke)

In [8]:
import { HumanMessage } from "langchain";

const result = await agent.invoke({
    messages: [new HumanMessage("Tell me a joke")]
})

console.log(result.messages.at(-1).content)

Why don't scientists trust atoms?

Because they make up everything! 

🔬⚛️


Notice how we had to wait for the complete response. Let's improve the user experience with streaming.


## Streaming
### `value`

In [9]:
const stream = await agent.stream({
    messages: [new HumanMessage("Tell me a joke")],
}, {
    streamMode: "values",
})

for await (const step of stream) {
    console.log(step.messages.at(-1).content)
}

Tell me a joke
Why don't scientists trust atoms?

Because they make up everything! 

🔬⚛️


The `values` mode streams the entire state at each step. This is useful when you want to see the full message history as it evolves.

### `messages`

For a more granular experience, `messages` mode streams individual message chunks as they're generated - perfect for real-time token-by-token display.


In [12]:
const stream = await agent.stream({
    messages: [new HumanMessage("Tell me a joke")],
}, {
    streamMode: "messages",
})

for await (const [message, metadata] of stream) {
    console.log(`[${metadata.langgraph_node}]: ${message.content}`)
}

[model_request]: 
[model_request]: 
[model_request]: Here's one
[model_request]:  for you:

A SQL query walks into a bar, walks up to two
[model_request]:  tables and asks...

**"Can I JOIN you?"**

🥁
[model_request]:  *ba dum tss*

The tables said no. They
[model_request]:  had **foreign key** issues from a past
[model_request]:  relationship. 😄

Want another one? I've
[model_request]:  got jokes in every stack — front end, back end, and
[model_request]:  the dark corners of CSS! 😂
[model_request]: 


Here's a more dramatic example - streaming a poem token by token creates a typewriter effect:


In [5]:
await Deno.jupyter.broadcast("display_data", {
    data: { "text/markdown": "🤔" },
    metadata: {},
    transient: { display_id: "progress" }
});

const stream = await agent.stream({
    messages: [new HumanMessage("Write me a poem.")],
}, {
    streamMode: "messages",
})

let content = "";
const i = setInterval(async () => {
    Deno.jupyter.broadcast("update_display_data", {
        data: { "text/markdown": content },
        metadata: {},
        transient: { display_id: "progress" }
    });
}, 1)

for await (const [message, metadata] of stream) {
    content += message.content
}

clearInterval(i)

# Ode to My Phone Battery

My battery was at a hundred percent,
A glorious sight, a gift from the heavens sent.
I unplugged it, proud, stepped out the door,
Then looked down and saw... **fourteen percent. No more.**

I hadn't even *texted* yet.
I hadn't Googled one single thing.
I simply *existed* near the phone
And watched it suffer, perishing.

I plugged it in at nine o'clock,
And waited, hopeful, full of grace.
By midnight it hit thirty-two.
The charger looked me in the face

And said, *"You bought me at a gas station.
For four dollars. What did you expect?"*

Fair point.

I turned the brightness down to zero,
Turned off location, Bluetooth, sound.
The phone said **eleven percent**
And somehow... kept going down.

So here I lie in darkened silence,
One bar of signal, screen gone black.
At **two percent** I'll text my loved ones.
At **one percent** — I won't look back.

🪦 *Here lies my phone. It tried.*

## Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A stream writer allows you to easily stream `custom` data from sources you create.

In [6]:
import * as setup from "./setup.ts";
import { z } from "zod";
import { createAgent, tool, type Runtime } from "langchain";
import { AsyncLocalStorageProviderSingleton } from "npm:@langchain/core/singletons";

const getWeather = tool(({ city }, runtime: Runtime) => {
    runtime.writer(`Looking up data for city: ${city}`);
    runtime.writer(`Acquired data for city: ${city}`);
    return `It's always sunny in ${city}`
}, {
    name: "get_weather",
    description: "Get weather for a given city.",
    schema: z.object({
        city: z.string()
    })
});

const toolCallingAgent = createAgent({
    model: process.env.AI_MODEL || "anthropic:claude-sonnet-4-6",
    tools: [getWeather]
})

// Note: Deno Jupyter requires wrapping stream() in a clean async context
const stream = await AsyncLocalStorageProviderSingleton.runWithConfig(
    {},
    () => toolCallingAgent.stream({
        messages: "What's the weather in SF?",
    }, {
        streamMode: ["values", "custom"],
    })
);

for await (const [type, stateOrCustomEvent] of stream) {
    if (type === "values") {
        displayMessage(stateOrCustomEvent.messages.at(-1))
    } else if (type === "custom") {
        displayMessage({
            type,
            content: stateOrCustomEvent
        })
    }
}


┌────────────────────────────────────────────────────────────┐
│ 👤 HUMAN MESSAGE                                           │
└────────────────────────────────────────────────────────────┘
What's the weather in SF?

┌────────────────────────────────────────────────────────────┐
│ 🤖 AI MESSAGE                                              │
└────────────────────────────────────────────────────────────┘
[
  {
    type: "tool_use",
    id: "toolu_01L2LjtDiyL6K4NDmwKVXUEu",
    name: "get_weather",
    input: { city: "San Francisco" },
    caller: { type: "direct" }
  }
]

┌────────────────────────────────────────────────────────────┐
│ 💡 CUSTOM MESSAGE                                          │
└────────────────────────────────────────────────────────────┘
Looking up data for city: San Francisco

┌────────────────────────────────────────────────────────────┐
│ 💡 CUSTOM MESSAGE                                          │
└────────────────────────────────────────────────────────────┘
Acquired

## Try your own.
Create a tool of your own and try it here!

In [7]:
// Note: same workaround needed here
const stream2 = await AsyncLocalStorageProviderSingleton.runWithConfig(
    {},
    () => toolCallingAgent.stream({
        messages: "What's the weather in SF?",
    }, {
        streamMode: ["values", "custom"],
    })
);

for await (const [type, chunk] of stream2) {
    if (type === "custom") {
        console.log(chunk)
    }
}

Looking up data for city: SF


Acquired data for city: SF
